In [ ]:
from abc import ABC, abstractmethod
from enum import Enum
from typing import Any, List, Optional

# Enum representing your 3 calculation scenarios
class EvaluationMode(Enum):
    TRANSFORMED_SIMPLE = 1   # Transformed inputs + Simple average
    DIRECT_WEIGHTED = 2      # Direct score inputs + Weighted average
    TRANSFORMED_WEIGHTED = 3 # Transformed inputs + Weighted average

class TransformationStrategy(ABC):
    @abstractmethod
    def evaluate(self, raw_input: Any) -> float:
        pass


class ModelElement(ABC):
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def calculate_score(self, context: Dict[str, Any], mode: EvaluationMode) -> float:
        pass


class ProcedureElement(ModelElement):
    def __init__(self, name: str, default_strategy: TransformationStrategy):
        super().__init__(name)
        self.default_strategy = default_strategy

    def calculate_score(self, context: Dict[str, Any], mode: EvaluationMode) -> float:
        if mode == EvaluationMode.DIRECT_WEIGHTED:
            # DIRECT mode: Read straight from user-provided quality metrics
            quality_metrics = context.get("quality_metric_inputs", {})
            if self.name not in quality_metrics:
                raise KeyError(f"Missing quality metric input for procedure: '{self.name}'")
            return float(quality_metrics[self.name])
        
        else:
            # TRANSFORMED modes (1 & 3): Read raw procedure inputs and transform them
            procedure_inputs = context.get("procedure_inputs", {})
            if self.name not in procedure_inputs:
                raise KeyError(f"Missing raw procedure input for procedure: '{self.name}'")
            raw_data = procedure_inputs[self.name]
            return self.default_strategy.evaluate(raw_data)


class CompositeElement(ModelElement):
    def __init__(
        self, 
        name: str, 
        subelements: Optional[List[ModelElement]] = None,
        default_weights: Optional[List[float]] = None
    ):
        super().__init__(name)
        self.subelements = subelements or []
        self.default_weights = default_weights

    def calculate_score(self, context: Dict[str, Any], mode: EvaluationMode) -> float:
        if not self.subelements:
            return 0.0

        # Recursively get child scores using the current mode
        child_scores = [child.calculate_score(context, mode) for child in self.subelements]

        # Mode 1: Simple average (equal weights)
        if mode == EvaluationMode.TRANSFORMED_SIMPLE:
            return sum(child_scores) / len(child_scores)

        # Modes 2 & 3: Weighted average
        # Get custom weights from runtime context, or fallback to default weights, or equal weights
        weights = context.get("weights", {}).get(self.name) or self.default_weights
        if not weights or len(weights) != len(child_scores):
            weights = [1.0] * len(child_scores)

        return sum(s * w for s, w in zip(child_scores, weights)) / sum(weights)

# --- Concrete Rule (Library Internal) ---
class DostepnoscUslugiTransformation(TransformationStrategy):
    def evaluate(self, raw_input: Any) -> float:
        val = float(raw_input)
        if val < 0.95: return 1.0
        elif val < 0.99: return 2.0
        elif val < 0.999: return 3.0
        elif val < 0.9999: return 4.0
        else: return 5.0


# --- Pre-packaged Model Factory (Library Internal) ---
def load_standard_quality_model() -> CompositeElement:
    """Pre-built 20+ element hierarchy shipped inside the library."""
    
    # 1. Standard Procedures
    proc_dostepnosc = ProcedureElement("Dostępność Usługi", DostepnoscUslugiTransformation())
    
    # Imagine 20+ other procedures defined here...
    
    # 2. Standard Characteristics / Sub-models
    niezawodnosc = CompositeElement(
        name="Niezawodność Usługi",
        subelements=[proc_dostepnosc],
        default_weights=[1.0]
    )

    # 3. Root Model
    root_model = CompositeElement(
        name="System Quality Model",
        subelements=[niezawodnosc],
        default_weights=[1.0]
    )
    
    return root_model